# 🏗️ SAM 3 Building Counter — Inference Server

This notebook runs Meta's SAM 3 model on a **free Google Colab GPU** and exposes it as an API for the GIS Building Counter dashboard.

## Instructions
1. Make sure the runtime is set to **GPU** (Runtime → Change runtime type → T4 GPU)
2. Click **Runtime → Run all** (or Ctrl+F9)
3. Copy the public URL printed at the bottom
4. Paste it into the dashboard **Settings** modal

> ⚠️ The free Colab session will stay active for ~12 hours. You can re-run this notebook anytime to get a new URL.

## Step 1 — Install Dependencies

In [ ]:
# Check GPU is available
!nvidia-smi
print('\n✅ GPU detected' if '0' in open('/proc/driver/nvidia/gpus/0000:00:04.0/information', 'r').read() else '\n⚠️ No GPU — change runtime to T4 GPU')

In [ ]:
%%capture
# Install SAM 3 and server dependencies
!pip install -q torch torchvision
!pip install -q git+https://github.com/facebookresearch/sam3.git
!pip install -q flask flask-cors
!pip install -q pyngrok
!apt-get -qq install -y cloudflared > /dev/null 2>&1 || true
print('✅ Dependencies installed')

## Step 2 — Authenticate with Hugging Face

SAM 3 model weights are hosted on Hugging Face. You need a free account:
1. Create an account at [huggingface.co](https://huggingface.co/join)
2. Go to [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
3. Create a new token with **Read** access
4. Paste it below when prompted

In [ ]:
from huggingface_hub import login
login()

## Step 3 — Start the Inference Server

This cell starts the Flask server and creates a public tunnel. The URL will be printed below — copy it into the dashboard Settings.

In [ ]:
import io
import time
import json
import logging
import subprocess
import re
from threading import Thread

import torch
import numpy as np
from PIL import Image
from flask import Flask, request, jsonify
from flask_cors import CORS

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# --- Model Loading ---
print('📦 Loading SAM 3 model (this takes 1-2 minutes on first run)...')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'   Device: {device}')

try:
    from sam3.build_sam import sam_model_registry
    model = sam_model_registry['sam3_hiera_large'](checkpoint=None)
    model = model.to(device).eval()
    print('✅ SAM 3 model loaded')
    MODEL_LOADED = True
except Exception as e:
    print(f'⚠️ SAM 3 model loading failed: {e}')
    print('   The server will still start but return simulated results.')
    print('   This is useful for testing the dashboard connection.')
    MODEL_LOADED = False

# --- Flask Server ---
app = Flask(__name__)
CORS(app)


@app.route('/health', methods=['GET'])
def health():
    return jsonify({
        'status': 'healthy',
        'model_loaded': MODEL_LOADED,
        'device': device,
        'gpu_available': torch.cuda.is_available(),
        'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    })


@app.route('/predict', methods=['POST'])
def predict():
    start_time = time.time()

    if 'image' not in request.files:
        return jsonify({'error': 'No image provided'}), 400

    image_file = request.files['image']
    image = Image.open(image_file.stream).convert('RGB')
    width, height = image.size

    prompt = request.form.get('prompt', 'building')
    confidence_threshold = float(request.form.get('confidence_threshold', 0.5))
    min_area = request.form.get('min_area')
    max_area = request.form.get('max_area')
    if min_area: min_area = int(min_area)
    if max_area: max_area = int(max_area)

    logger.info(f'Processing {width}x{height} image with prompt: "{prompt}"')

    if MODEL_LOADED:
        try:
            # Run actual SAM 3 inference
            img_array = np.array(image)
            # SAM 3 inference pipeline here
            # This will be updated once we confirm the exact API
            from sam3.automatic_mask_generator import SamAutomaticMaskGenerator
            mask_generator = SamAutomaticMaskGenerator(model)
            masks = mask_generator.generate(img_array)

            boxes = []
            scores = []
            mask_data = []

            for m in masks:
                bbox = m['bbox']  # [x, y, w, h]
                box = [bbox[0], bbox[1], bbox[0]+bbox[2], bbox[1]+bbox[3]]
                score = m.get('predicted_iou', m.get('stability_score', 0.8))
                area = bbox[2] * bbox[3]

                if score < confidence_threshold:
                    continue
                if min_area and area < min_area:
                    continue
                if max_area and max_area > 0 and area > max_area:
                    continue

                boxes.append(box)
                scores.append(float(score))
                mask_data.append({
                    'area': int(m['area']),
                    'size': [height, width],
                })

            processing_time = time.time() - start_time
            logger.info(f'Found {len(boxes)} objects in {processing_time:.2f}s')

            return jsonify({
                'masks': mask_data,
                'boxes': boxes,
                'scores': scores,
                'processing_time': round(processing_time, 2),
                'image_width': width,
                'image_height': height,
            })

        except Exception as e:
            logger.error(f'Inference error: {e}')
            return jsonify({'error': str(e)}), 500
    else:
        # Simulated response for testing the dashboard
        import random
        num_buildings = random.randint(5, 30)
        boxes = []
        scores = []
        mask_data = []

        for i in range(num_buildings):
            x1 = random.randint(0, width - 60)
            y1 = random.randint(0, height - 60)
            w = random.randint(20, 80)
            h = random.randint(20, 80)
            score = random.uniform(0.5, 0.98)

            if score < confidence_threshold:
                continue

            boxes.append([x1, y1, x1+w, y1+h])
            scores.append(round(score, 4))
            mask_data.append({'area': w*h, 'size': [height, width]})

        processing_time = time.time() - start_time
        return jsonify({
            'masks': mask_data,
            'boxes': boxes,
            'scores': scores,
            'processing_time': round(processing_time, 2),
            'image_width': width,
            'image_height': height,
            'note': 'Simulated results — model not loaded',
        })


# --- Tunnel ---
def start_tunnel():
    time.sleep(2)  # Wait for Flask to start
    try:
        proc = subprocess.Popen(
            ['cloudflared', 'tunnel', '--url', 'http://localhost:5000', '--no-autoupdate'],
            stdout=subprocess.PIPE, stderr=subprocess.PIPE,
        )
        for line in proc.stderr:
            line = line.decode()
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
            if match:
                url = match.group()
                print(f'\n{"="*60}')
                print(f'  🌐 SAM 3 API is live at:')
                print(f'  {url}')
                print(f'\n  📋 Copy this URL and paste it into')
                print(f'     the dashboard Settings modal')
                print(f'{"="*60}\n')
                return
    except FileNotFoundError:
        pass

    # Fallback: ngrok
    try:
        from pyngrok import ngrok
        tunnel = ngrok.connect(5000)
        url = tunnel.public_url
        print(f'\n{"="*60}')
        print(f'  🌐 SAM 3 API is live at:')
        print(f'  {url}')
        print(f'\n  📋 Copy this URL and paste it into')
        print(f'     the dashboard Settings modal')
        print(f'{"="*60}\n')
    except Exception as e:
        print(f'⚠️ Could not create tunnel: {e}')
        print('   Server running at http://localhost:5000 (local only)')


# Start tunnel in background
Thread(target=start_tunnel, daemon=True).start()

# Start Flask (this blocks)
print('🚀 Starting SAM 3 inference server on port 5000...')
app.run(host='0.0.0.0', port=5000, debug=False)